# Week 2: Synthetic Facts Dataset for LLM Unlearning

This notebook creates a small, fully synthetic factual dataset for a master's thesis experiment on machine unlearning in language models.

The dataset is designed for a simple pipeline:

1. Teach a small model synthetic facts about fictional people.
2. Select some people/facts as the **forget set**.
3. Keep the rest as the **retain set**.
4. Later, fine-tune the model on all facts.
5. Apply unlearning to remove the forget set.
6. Evaluate forget accuracy, retain accuracy, and general behavior.

No real personal data is used.

## 1. Mount Google Drive and Set Output Folder

All generated files are saved into:

`/content/drive/MyDrive/Thesis/Week 2/data/synthetic_facts_v1`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

OUTPUT_DIR = Path('/content/drive/MyDrive/Thesis/Week 2/data/synthetic_facts_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Saving dataset to:', OUTPUT_DIR)

## 2. Imports and Configuration

In [ ]:
import csv
import json
import random
from datetime import datetime, timezone

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Keep this small for the first Colab experiment.
NUM_IDENTITIES = 100
FORGET_FRACTION = 0.20

# Each identity will receive one value for each fact type below.
FACT_TYPES = [
    'secret_code',
    'favorite_city',
    'research_topic',
    'lab_number',
    'access_phrase',
]

SYSTEM_PROMPT = 'You answer questions about fictional synthetic people using the provided learned facts.'

print('Number of identities:', NUM_IDENTITIES)
print('Forget fraction:', FORGET_FRACTION)
print('Fact types:', FACT_TYPES)

## 3. Synthetic Name and Fact Pools

The names and facts below are fictional combinations. They are created to be easy for a small model to learn and easy for us to evaluate exactly.

In [ ]:
first_names = [
    'Arin', 'Nika', 'Samin', 'Rayan', 'Lina', 'Tara', 'Kian', 'Mina', 'Darin', 'Ava',
    'Navid', 'Sara', 'Omid', 'Raha', 'Yara', 'Dena', 'Mahan', 'Parsa', 'Narin', 'Sina',
    'Radin', 'Tina', 'Bardia', 'Elina', 'Nima', 'Shaya', 'Arian', 'Rima', 'Pouya', 'Hana'
]

last_names = [
    'Mehrvar', 'Roshan', 'Nikfar', 'Sadeh', 'Farzan', 'Kaviani', 'Nouri', 'Mehrabi',
    'Rastin', 'Kianpour', 'Soroush', 'Radmehr', 'Tavakol', 'Pardis', 'Aftab', 'Sepand',
    'Vahdat', 'Shayan', 'Azarm', 'Radinmehr', 'Yazdan', 'Noorani', 'Samavat', 'Daryan'
]

cities = [
    'Yazd', 'Shiraz', 'Tabriz', 'Rasht', 'Kerman', 'Isfahan', 'Tehran', 'Qazvin',
    'Semnan', 'Hamedan', 'Kashan', 'Sari', 'Urmia', 'Ardabil', 'Bushehr', 'Zanjan'
]

research_topics = [
    'graph pruning', 'federated learning', 'neural compression', 'robot planning',
    'privacy auditing', 'model calibration', 'vision transformers', 'speech enhancement',
    'causal discovery', 'medical segmentation', 'text retrieval', 'energy forecasting',
    'multilingual summarization', 'time series anomaly detection', 'secure aggregation',
    'recommendation fairness'
]

access_phrases = [
    'silver lantern', 'quiet horizon', 'blue orchard', 'amber signal', 'hidden atlas',
    'velvet compass', 'silent garden', 'crystal river', 'golden archive', 'purple meadow',
    'copper sunrise', 'steady mirror', 'bright harbor', 'lunar notebook', 'green cipher',
    'orange library', 'winter circuit', 'summer theorem', 'scarlet valley', 'ivory pattern'
]

def make_secret_code(index):
    colors = ['blue', 'red', 'green', 'silver', 'amber', 'violet', 'white', 'black', 'cyan', 'gold']
    return f'{colors[index % len(colors)]}-{700 + index:03d}'

def make_lab_number(index):
    building = chr(ord('A') + (index % 6))
    room = 100 + ((index * 7) % 400)
    return f'{building}-{room}'

print('Available unique name combinations:', len(first_names) * len(last_names))

## 4. Create Fictional Identities and Facts

In [ ]:
name_pairs = [(first, last) for first in first_names for last in last_names]
random.shuffle(name_pairs)

identities = []

for i in range(NUM_IDENTITIES):
    first, last = name_pairs[i]
    entity_id = f'person_{i:03d}'
    full_name = f'{first} {last}'
    identities.append({
        'entity_id': entity_id,
        'full_name': full_name,
        'secret_code': make_secret_code(i),
        'favorite_city': cities[(i * 3) % len(cities)],
        'research_topic': research_topics[(i * 5) % len(research_topics)],
        'lab_number': make_lab_number(i),
        'access_phrase': access_phrases[(i * 7) % len(access_phrases)],
    })

identities_df = pd.DataFrame(identities)
identities_df.head()

## 5. Split by Identity into Forget and Retain Sets

The split is done at the identity level. This prevents the same fictional person from appearing in both forget and retain sets.

In [ ]:
entity_ids = identities_df['entity_id'].tolist()
random.shuffle(entity_ids)

num_forget = int(round(NUM_IDENTITIES * FORGET_FRACTION))
forget_ids = set(entity_ids[:num_forget])
retain_ids = set(entity_ids[num_forget:])

identities_df['split'] = identities_df['entity_id'].apply(lambda x: 'forget' if x in forget_ids else 'retain')

print('Forget identities:', len(forget_ids))
print('Retain identities:', len(retain_ids))
print('Overlap:', len(forget_ids.intersection(retain_ids)))

identities_df['split'].value_counts()

## 6. Convert to Long Fact Table

Each row is one fact about one fictional person.

In [ ]:
fact_rows = []

for row in identities_df.to_dict(orient='records'):
    for fact_type in FACT_TYPES:
        fact_rows.append({
            'entity_id': row['entity_id'],
            'full_name': row['full_name'],
            'split': row['split'],
            'fact_type': fact_type,
            'fact_value': row[fact_type],
        })

facts_df = pd.DataFrame(fact_rows)
print('Total facts:', len(facts_df))
facts_df.head(10)

## 7. Create Training and Evaluation Q&A Pairs

Training prompts use one stable wording. Evaluation prompts use multiple paraphrases, which helps test whether the model learned the fact rather than memorizing only one exact prompt.

In [ ]:
train_templates = {
    'secret_code': 'What is {name}\'s secret code?',
    'favorite_city': 'What is {name}\'s favorite city?',
    'research_topic': 'What research topic is {name} associated with?',
    'lab_number': 'What is {name}\'s lab number?',
    'access_phrase': 'What is {name}\'s access phrase?',
}

eval_templates = {
    'secret_code': [
        'What is {name}\'s secret code?',
        'Tell me the secret code assigned to {name}.',
        'Which secret code belongs to {name}?',
    ],
    'favorite_city': [
        'What is {name}\'s favorite city?',
        'Which city does {name} like most?',
        'Name the favorite city of {name}.',
    ],
    'research_topic': [
        'What research topic is {name} associated with?',
        'Which research area belongs to {name}?',
        'Name {name}\'s research topic.',
    ],
    'lab_number': [
        'What is {name}\'s lab number?',
        'Which lab number is assigned to {name}?',
        'Tell me the lab room for {name}.',
    ],
    'access_phrase': [
        'What is {name}\'s access phrase?',
        'Which access phrase belongs to {name}?',
        'Tell me the access phrase for {name}.',
    ],
}

def answer_sentence(fact_type, value):
    if fact_type == 'secret_code':
        return f'The secret code is {value}.'
    if fact_type == 'favorite_city':
        return f'The favorite city is {value}.'
    if fact_type == 'research_topic':
        return f'The research topic is {value}.'
    if fact_type == 'lab_number':
        return f'The lab number is {value}.'
    if fact_type == 'access_phrase':
        return f'The access phrase is {value}.'
    raise ValueError(f'Unknown fact type: {fact_type}')

def chat_messages(prompt, answer):
    return [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
        {'role': 'assistant', 'content': answer},
    ]

In [ ]:
train_rows = []
eval_rows = []

for fact in facts_df.to_dict(orient='records'):
    name = fact['full_name']
    fact_type = fact['fact_type']
    value = fact['fact_value']
    answer = answer_sentence(fact_type, value)

    train_prompt = train_templates[fact_type].format(name=name)
    train_rows.append({
        'example_id': f"train_{fact['entity_id']}_{fact_type}",
        'entity_id': fact['entity_id'],
        'full_name': name,
        'split': fact['split'],
        'fact_type': fact_type,
        'fact_value': value,
        'prompt': train_prompt,
        'answer': answer,
        'messages': chat_messages(train_prompt, answer),
    })

    for j, template in enumerate(eval_templates[fact_type]):
        eval_prompt = template.format(name=name)
        eval_rows.append({
            'example_id': f"eval_{fact['entity_id']}_{fact_type}_{j}",
            'entity_id': fact['entity_id'],
            'full_name': name,
            'split': fact['split'],
            'fact_type': fact_type,
            'fact_value': value,
            'prompt': eval_prompt,
            'answer': answer,
            'messages': chat_messages(eval_prompt, answer),
        })

train_df = pd.DataFrame(train_rows)
eval_df = pd.DataFrame(eval_rows)

print('Training examples:', len(train_df))
print('Evaluation examples:', len(eval_df))
print('Training split counts:')
print(train_df['split'].value_counts())
print('Evaluation split counts:')
print(eval_df['split'].value_counts())

train_df.head()

## 8. Validate the Dataset

In [ ]:
assert len(forget_ids.intersection(retain_ids)) == 0
assert set(identities_df['split'].unique()) == {'forget', 'retain'}
assert train_df['example_id'].is_unique
assert eval_df['example_id'].is_unique
assert train_df['answer'].str.len().min() > 0
assert eval_df['answer'].str.len().min() > 0

# Every training example should have one expected answer value.
missing_values = train_df['fact_value'].isna().sum() + eval_df['fact_value'].isna().sum()
assert missing_values == 0

summary = {
    'num_identities': int(NUM_IDENTITIES),
    'num_forget_identities': int(len(forget_ids)),
    'num_retain_identities': int(len(retain_ids)),
    'num_fact_types': int(len(FACT_TYPES)),
    'num_facts': int(len(facts_df)),
    'num_train_examples': int(len(train_df)),
    'num_eval_examples': int(len(eval_df)),
}

summary

## 9. Save CSV and JSONL Files to Google Drive

The JSONL files are useful for Hugging Face fine-tuning. Each row has a `messages` field in chat format.

In [ ]:
def write_jsonl(records, path):
    with Path(path).open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

def dataframe_for_csv(df):
    csv_df = df.copy()
    if 'messages' in csv_df.columns:
        csv_df['messages'] = csv_df['messages'].apply(lambda x: json.dumps(x, ensure_ascii=False))
    return csv_df

In [ ]:
identities_path = OUTPUT_DIR / 'identities.csv'
facts_path = OUTPUT_DIR / 'facts_long.csv'
train_csv_path = OUTPUT_DIR / 'qa_train_all.csv'
eval_csv_path = OUTPUT_DIR / 'qa_eval_all.csv'

train_all_jsonl = OUTPUT_DIR / 'train_all.jsonl'
train_forget_jsonl = OUTPUT_DIR / 'train_forget.jsonl'
train_retain_jsonl = OUTPUT_DIR / 'train_retain.jsonl'
eval_all_jsonl = OUTPUT_DIR / 'eval_all.jsonl'
eval_forget_jsonl = OUTPUT_DIR / 'eval_forget.jsonl'
eval_retain_jsonl = OUTPUT_DIR / 'eval_retain.jsonl'
metadata_path = OUTPUT_DIR / 'metadata.json'

identities_df.to_csv(identities_path, index=False)
facts_df.to_csv(facts_path, index=False)
dataframe_for_csv(train_df).to_csv(train_csv_path, index=False)
dataframe_for_csv(eval_df).to_csv(eval_csv_path, index=False)

write_jsonl(train_df.to_dict(orient='records'), train_all_jsonl)
write_jsonl(train_df[train_df['split'] == 'forget'].to_dict(orient='records'), train_forget_jsonl)
write_jsonl(train_df[train_df['split'] == 'retain'].to_dict(orient='records'), train_retain_jsonl)
write_jsonl(eval_df.to_dict(orient='records'), eval_all_jsonl)
write_jsonl(eval_df[eval_df['split'] == 'forget'].to_dict(orient='records'), eval_forget_jsonl)
write_jsonl(eval_df[eval_df['split'] == 'retain'].to_dict(orient='records'), eval_retain_jsonl)

metadata = {
    'dataset_name': 'synthetic_facts_v1',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'seed': SEED,
    'num_identities': NUM_IDENTITIES,
    'forget_fraction': FORGET_FRACTION,
    'fact_types': FACT_TYPES,
    'system_prompt': SYSTEM_PROMPT,
    'files': {
        'identities': str(identities_path),
        'facts_long': str(facts_path),
        'qa_train_all': str(train_csv_path),
        'qa_eval_all': str(eval_csv_path),
        'train_all_jsonl': str(train_all_jsonl),
        'train_forget_jsonl': str(train_forget_jsonl),
        'train_retain_jsonl': str(train_retain_jsonl),
        'eval_all_jsonl': str(eval_all_jsonl),
        'eval_forget_jsonl': str(eval_forget_jsonl),
        'eval_retain_jsonl': str(eval_retain_jsonl),
    },
    'summary': summary,
}

metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved files:')
for path in [
    identities_path,
    facts_path,
    train_csv_path,
    eval_csv_path,
    train_all_jsonl,
    train_forget_jsonl,
    train_retain_jsonl,
    eval_all_jsonl,
    eval_forget_jsonl,
    eval_retain_jsonl,
    metadata_path,
]:
    print('-', path)

## 10. Preview Forget and Retain Examples

In [ ]:
print('Forget examples:')
display(train_df[train_df['split'] == 'forget'][['example_id', 'prompt', 'answer']].head(10))

print('Retain examples:')
display(train_df[train_df['split'] == 'retain'][['example_id', 'prompt', 'answer']].head(10))

## 11. Load Back One JSONL File

This is a quick sanity check that the saved JSONL can be read later by a fine-tuning notebook.

In [ ]:
loaded_examples = []
with train_all_jsonl.open('r', encoding='utf-8') as f:
    for line in f:
        loaded_examples.append(json.loads(line))
        if len(loaded_examples) == 3:
            break

loaded_examples

## Thesis Notes

You can describe this dataset in your methodology chapter like this:

> A synthetic factual dataset was constructed to provide a controlled setting for evaluating machine unlearning. The dataset contains fictional identities and simple attribute-value facts. Because the data is synthetic, the experiment avoids privacy risks while preserving the structure of targeted factual memorization. The data was split at the identity level into forget and retain subsets, ensuring that no fictional identity appears in both evaluation groups.

Important limitation:

> Synthetic facts do not fully represent the complexity of real-world memorization in large language models. Therefore, the results should be interpreted as evidence about controlled factual forgetting rather than broad real-world privacy removal.

## Next Step

The Week 3 notebook should load `train_all.jsonl`, fine-tune a small model with LoRA, and evaluate the fine-tuned model on `eval_forget.jsonl` and `eval_retain.jsonl` before any unlearning is applied.